In [ ]:
import pandas as pd

df = pd.read_csv("/content/Final_MasterDataset.csv")


print("Total rows:", len(df))
print("\nMissing values per column:")
print(df.isna().sum())


Total rows: 155

Missing values per column:
Country                                                                       0
Year                                                                          0
D_Expenditure_GDP                                                             0
Conflict_Intensity                                                          155
Health_Expenditure_(% of GDP)                                                39
Education_Expenditure_(% of GDP)                                             30
Environmental_impact(CO2e/capita)                                             5
PM2.5 air pollution, mean annual exposure (micrograms per cubic meter)\n      0
Refugees                                                                      0
Asylum Seekers                                                                0
Total number of deaths                                                        0
Population                                                                  

In [ ]:
df = df.dropna(subset=["Environmental_impact(CO2e/capita)"])

print("Rows after dropping missing target:", len(df))


Rows after dropping missing target: 150


In [ ]:
train_df = df[df["Year"] <= 2017]
test_df  = df[df["Year"] > 2017]

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))


Train rows: 120
Test rows: 30


In [ ]:
FEATURES = ["Country", "Year", "D_Expenditure_GDP"]
TARGET = "Environmental_impact(CO2e/capita)"

X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_test = test_df[FEATURES]
y_test = test_df[TARGET]


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

cat_features = ["Country"]
num_features = ["Year", "D_Expenditure_GDP"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, num_features),
    ("cat", categorical_pipeline, cat_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", Ridge(alpha=1.0))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", round(mae, 4))
print("RMSE:", round(rmse, 4))
print("R2:", round(r2, 4))


MAE: 2.1791
RMSE: 2.473
R2: 0.6426


In [ ]:
# Extract feature names after preprocessing
feature_names = model.named_steps["preprocessor"].get_feature_names_out()

coefficients = model.named_steps["regressor"].coef_

for name, coef in zip(feature_names, coefficients):
    print(f"{name}: {coef:.4f}")


num__Year: -0.2033
num__D_Expenditure_GDP: 1.2822
cat__Country_FRA: -0.0811
cat__Country_GBR: 1.7139
cat__Country_RUS: 3.4309
cat__Country_USA: 9.7534


In [ ]:
FEATURES_BASE = ["Country", "Year"]

X_train_base = train_df[FEATURES_BASE]
X_test_base = test_df[FEATURES_BASE]

from sklearn.pipeline import Pipeline

preprocessor_base = ColumnTransformer([
    ("num", StandardScaler(), ["Year"]),
    ("cat", OneHotEncoder(drop="first"), ["Country"])
])

model_base = Pipeline([
    ("preprocessor", preprocessor_base),
    ("regressor", Ridge(alpha=1.0))
])

model_base.fit(X_train_base, y_train)
y_pred_base = model_base.predict(X_test_base)

from sklearn.metrics import r2_score
print("Baseline R2 (No Military):", r2_score(y_test, y_pred_base))


Baseline R2 (No Military): 0.5740544021607531
